In [1]:
import numpy as np
from xgboost import XGBRegressor 
from sklearn.metrics import r2_score 
from data_processor import DataReader, DataPrep
from sklearn.model_selection import GridSearchCV
from cv_generator import train_val_split, ExpandingWindowCV
from sklearn.ensemble import RandomForestRegressor
from model_eval import *
import pandas as pd
from scipy import stats
from sklearn.preprocessing import LabelEncoder
from functools import partial
from lightgbm import LGBMRegressor
import warnings


In [2]:
daily_data_path = r'data/daily_data'
intraday_data_path = r'data/intraday_data'
intraday_df = DataReader.read_intraday_data(intraday_data_path)
daily_df = DataReader.read_daily_data(daily_data_path)
# daily_df.to_pickle('daily_df.pkl')
# intraday_df.to_pickle('intraday_df.pkl')
# daily_df = pd.read_pickle('daily_df.pkl')
# intraday_df =  pd.read_pickle('intraday_df.pkl')
data_prep = DataPrep(intraday_df, daily_df)
target_df = data_prep.get_target(clip_MAD=True, normalize= True)
X_df = data_prep.get_features()#indicators= {'RSI_14':partial(ta.rsi, length= 14), 'RSI_5':partial(ta.rsi, length= 5)})
input_data = X_df.join(target_df[['y', 'y_actual_clipped']], how = 'right')

Reading intraday data up to 99991231
Reading intraday data up to 99991231


In [21]:
daily_df

SYMBOL   MIC  FREE_FLOAT_PERCENTAGE  EST_VOL  \
Date       Id                                                          
2010-01-04 BBG000MQ1SN9    DVA  XNYS                99.6464  0.13074   
           BBG000BBCQD7    SLM  XNYS                99.1117  0.31610   
           BBG000BBB3K1    RAI  XNYS                57.7643  0.11253   
           BBG000BNMHS4    MAN  XNYS                99.3401  0.18568   
           BBG000C23PB0    SAI  XNYS                99.1347  0.11252   
...                        ...   ...                    ...      ...   
2014-12-31 BBG000BPH459   MSFT  XNGS                91.5633  0.12146   
           BBG000GZQ728    XOM  XNYS                99.7689  0.09480   
           BBG000CKGBP2   GILD  XNGS                99.4121  0.20535   
           BBG000MM2P62     FB  XNGS                95.9988  0.17343   
           BBG000B9XRY4   AAPL  XNGS                99.9441  0.16047   

                               MDV_63    Open    High     Low   Close  \
Date       Id                                                           
2010-01-04 BBG000MQ1SN9  4.383275e+07   59.12   60.05   59.09   59.92   
           BBG000BBCQD7  4.409296e+07   11.45   11.72   11.32   11.54   
           BBG000BBB3K1  7.588626e+07   53.33   53.52   53.00   53.24   
           BBG000BNMHS4  4.413393e+07   54.94   56.58   54.78   56.41   
           BBG000C23PB0  4.440541e+07   19.02   19.17   18.89   19.11   
...                               ...     ...     ...     ...     ...   
2014-12-31 BBG000BPH459  1.383197e+09   46.73   47.44   46.45   46.45   
           BBG000GZQ728  1.273231e+09   92.42   93.13   92.06   92.45   
           BBG000CKGBP2  1.589295e+09   96.00   96.75   94.24   94.26   
           BBG000MM2P62  2.258944e+09   79.54   79.80   77.86   78.02   
           BBG000B9XRY4  5.093251e+09  112.82  113.13  110.21  110.38   

                             Volume  PxAdjFactor  SharesAdjFactor  
Date       Id                                                      
2010-01-04 BBG000MQ1SN9    955120.0     1.000000         1.000000  
           BBG000BBCQD7   2566745.0     1.000000         1.000000  
           BBG000BBB3K1    812905.0     1.160364         1.000000  
           BBG000BNMHS4   1108030.0     1.034430         1.000000  
           BBG000C23PB0   3500468.0     1.000000         1.000000  
...                             ...          ...              ...  
2014-12-31 BBG000BPH459  21551092.0     1.188881         1.000000  
           BBG000GZQ728  11326179.0     1.188282         1.000000  
           BBG000CKGBP2  13851283.0     2.000000         0.500000  
           BBG000MM2P62  20004654.0     1.000000         1.000000  
           BBG000B9XRY4  41304130.0     7.391928         0.142857  

[629000 rows x 12 columns]

In [20]:
intraday_df.sort_index(level= 'Date')

Time  CumReturnResid  CumReturnRaw  CumVolume
Date       Id                                                             
2010-01-04 BBG000B9WH86  09:45:00       -0.003154      0.021705    2030006
           BBG000B9WH86  10:00:00       -0.002442      0.024186    3955351
           BBG000B9WH86  10:15:00        0.006524      0.037209    6260704
           BBG000B9WH86  10:30:00        0.009743      0.043411    8236062
           BBG000B9WH86  10:45:00        0.005682      0.041551    9894913
...                           ...             ...           ...        ...
2014-12-31 BBG006B6PVN9  15:00:00        0.010571      0.005199     173925
           BBG006B6PVN9  15:15:00        0.010221      0.005694     180767
           BBG006B6PVN9  15:30:00        0.012920      0.008417     195729
           BBG006B6PVN9  15:45:00        0.016027      0.009655     250247
           BBG006B6PVN9  16:00:00        0.018198      0.009160     408596

[16287206 rows x 4 columns]

### Train Val Split

In [3]:
features = ['CumReturnResid',  'Rolling_Return_5d', 'Rolling_Return_10d', 'Rolling_Return_20d','NYSE', 'IntradayRSI']#, 'RSI_14']#,'Stock_Split', 'Dividend', 'Rolling_Return_20d', 'EarlyClose', 'NextHoliday']
clipped_returns = [col for col in input_data.columns if 'clipped' in col and 'Return' in col]
features = [f'Rolling_Return_{i}d_clipped' for i in [5, 10]]  + ['CumReturnResid', 'IntradayRSI', 'NYSE']# 'VolumeChangeNormalize']#, 'NYSE'] + 
input_data.dropna(subset=features, inplace=True)
train_data, val_data = train_val_split(input_data, 0.8)
print(features)
x_train, y_train = train_data[features], train_data['y']
x_val, y_val = val_data[features], val_data['y_actual_clipped']#val_data['y']
train_weights = train_data.MDV_63_sqrt.to_numpy()
val_weights = val_data.MDV_63_sqrt.to_numpy()

['Rolling_Return_5d_clipped', 'Rolling_Return_10d_clipped', 'CumReturnResid', 'IntradayRSI', 'NYSE']


In [ ]:
train_data[clipped_returns + ['y']].corr()['y'].sort_values()

In [ ]:
train_data[features + ['y']].corr()['y'].sort_values()


### Base random forest grid search

In [ ]:
rf = RandomForestRegressor()

rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth': [2, 4, 6, 10],
    'min_samples_leaf': [2, 5, 10]
}

expand_wind_rf = ExpandingWindowCV(rf, rf_param_grid) 
expand_wind_rf.fit(x_train, y_train, train_weights)
print(expand_wind_rf.grid_search.best_params_)
model_rf = ModelEval(RandomForestRegressor(),features, x_train, y_train, train_weights, x_val.to_numpy(), y_val.to_numpy(), val_data.EST_VOL_preday.to_numpy(), val_weights,\
    grid_search= expand_wind_rf.grid_search)   
model_rf.to_pickle('xgboost_iter1')
print(model_rf.weighted_r2())
print(model_rf.feature_importance())
print(model_rf.feature_importance_MDA())

### Base xgboost Grid Search

In [ ]:
xgb = XGBRegressor()
xgb_param_grid = {
    'max_depth': [2, 4, 6],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.7, 1.0],
    'min_child_weight': [1, 5, 10],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

expand_wind_xgb = ExpandingWindowCV(xgb, xgb_param_grid) 
expand_wind_xgb.fit(x_train, y_train, train_weights)
print(expand_wind_xgb.grid_search.best_params_)
# best_param = {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 100, 'reg_alpha': 1, 'reg_lambda': 2, 'subsample': 0.7}
model = ModelEval(XGBRegressor(),features, x_train, y_train, train_weights, x_val.to_numpy(), y_val.to_numpy(), val_data.EST_VOL_preday.to_numpy(), val_weights, grid_search= expand_wind_xgb.grid_search)
# model.to_pickle('xgboost_base')
print(model.weighted_r2())
print(model.feature_importance())
print(model.feature_importance_MDA())

### Base LightGBM Grid Search

In [16]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(verbosity = -1)

lgbm_param_grid = {
    'max_depth': [2, 4, 6],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 150],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.7, 1.0],
    'min_child_samples': [1, 5, 10],
    'lambda_l2': [0.05, 0.1, 0.2, 0.3]
}

# expand_wind_lgbm = ExpandingWindowCV(lgbm, lgbm_param_grid) 
# expand_wind_lgbm.fit(x_train, y_train, train_weights)
# model_lgbm = ModelEval(LGBMRegressor(verbosity = -1),features, x_train, y_train, train_weights, x_val.to_numpy(), y_val.to_numpy(), val_data.EST_VOL_preday.to_numpy(), val_weights, expand_wind_lgbm.grid_search)
# model_lgbm.to_pickle('lgbm_iter2')
# model_lgbm = load_model('lgbm_iter1')
# print(model_lgbm.grid_cv.best_params_)
best_params = {'colsample_bytree': 1.0, 'lambda_l1': 0.3, 'learning_rate': 0.01, 'max_depth': 4, 'min_child_samples': 5, 'n_estimators': 100, 'subsample': 0.7}
model_lgbm = ModelEval(LGBMRegressor(verbosity = -1),features, x_train, y_train, train_weights, x_val.to_numpy(), y_val.to_numpy(), \
    val_data.EST_VOL_preday.to_numpy(), val_weights,best_params= best_params)
print(model_lgbm.weighted_r2())
print(model_lgbm.feature_importance())
print(model_lgbm.feature_importance_MDA())

ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: NYSE: object

### Final Model

In [14]:
x_train_all, y_train_all = input_data[features], input_data['y']
train_all_weights = input_data.MDV_63_sqrt.to_numpy()
best_params = {'colsample_bytree': 1.0, 'lambda_l1': 0.3, 'learning_rate': 0.01, 'max_depth': 4, 'min_child_samples': 5, 'n_estimators': 100, 'subsample': 0.7}
lgbm_final = LGBMRegressor(verbosity = -1, ** best_params)
lgbm_final.fit(x_train_all, y_train_all, sample_weight=train_all_weights)

LGBMRegressor(lambda_l1=0.3, learning_rate=0.01, max_depth=4,
              min_child_samples=5, subsample=0.7, verbosity=-1)

In [15]:
# Open the file in binary-write mode and dump the model
with open(r'final_model\lgbm_model.pkl', 'wb') as file:
    pickle.dump(lgbm_final, file)